<a href="https://colab.research.google.com/github/pop123-ux/DeepSeek-v4-pro-Implementation-using-Gradio/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💬 DeepSeek Chat — Gradio UI

A streaming chat interface over DeepSeek's OpenAI-compatible API, built with Gradio and runnable straight from Colab.

**Before running:** set your DeepSeek API key.
- **In Colab:** open the 🔑 **Secrets** panel (left sidebar) and add `DEEPSEEK_API_KEY` — the notebook picks it up automatically.
- **Locally:** export `DEEPSEEK_API_KEY` in your shell before launching Jupyter, or enter it at the prompt when the auth cell runs.

## 1. Install dependencies

In [ ]:
%pip install --quiet gradio openai

## 2. Load the API key

Prefers Colab's Secrets store; falls back to the `DEEPSEEK_API_KEY` env var; finally prompts interactively.

In [ ]:
import os

api_key = os.getenv('DEEPSEEK_API_KEY')

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('DEEPSEEK_API_KEY')
    except Exception:
        pass

if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your DeepSeek API key: ')

os.environ['DEEPSEEK_API_KEY'] = api_key
print('API key loaded ✅')

## 3. Wire up the client and streaming chat function

In [ ]:
from openai import OpenAI

MODEL = 'deepseek-v4-pro'
SYSTEM_PROMPT = 'You are a helpful, concise AI assistant.'

client = OpenAI(
    api_key=api_key,
    base_url='https://api.deepseek.com',
)

def stream_chat_api(message, history):
    """Stream DeepSeek's response for a single user turn.

    `history` from Gradio's ChatInterface (Gradio 5+) is already a list of
    {'role': ..., 'content': ...} dicts, so we prepend the system prompt
    and append the new user message.
    """
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + list(history)
    messages.append({'role': 'user', 'content': message})

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            stream=True,
        )
        partial_text = ''
        for chunk in response:
            delta = chunk.choices[0].delta.content
            if delta:
                partial_text += delta
                yield partial_text
    except Exception as e:
        yield f'**API error:** `{e}`'

## 4. Launch the Gradio UI

`share=True` gives you a public URL — handy in Colab where the inline iframe is cramped.

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=stream_chat_api,
    title='DeepSeek Chat',
    description=f'Model: `{MODEL}` · streaming responses',
    examples=[
        'Explain transformers in one paragraph.',
        'Write a Python function that reverses a string in place.',
        'What are three underrated data structures?',
    ],
)
demo.launch(share=True)